# Pilot 2025 Calibration, Estimability, and Natural-Must MBDoE

This notebook applies the reduced extended fermentation model to the pilot-scale natural-must dataset. It mirrors the laboratory workflow, but the design space is intentionally narrower: natural must is assumed, and the realistic manipulated inputs are temperature setpoints and nutrient additions.

The model combines primary fermentation, glycerol production, the reduced secondary v2 model, and empirical aroma synthesis for ethyl acetate, isoamyl acetate, and ethyl octanoate.

## Mathematical Structure

The primary state vector is

$$x_p = [X, X_d, N, G, F, E, Gly]^T$$

where `X` is viable biomass, `X_d` is dead biomass, `N` is assimilable nitrogen, `G` and `F` are glucose and fructose, `E` is ethanol, and `Gly` is glycerol.

The secondary model uses

$$x_s = [Pyr, AcAld, Acetate, O_2, CO_2]^T$$

with the reduced v2 free set

$$\theta_s = \{k_{PyrS,N}, k_{PyrO2}, k_{PyrDrain}, k_{AldS,N}, k_{AldRed}, k_{AcAld}, k_{AcStress}\}.$$

Aroma synthesis is empirical and rate-dependent:

$$r_i = \left(k_{i,growth}\phi_N + k_{i,stationary}(1-\phi_N)\right)q_S,$$

where `i` is ethyl acetate, isoamyl acetate, or ethyl octanoate; `q_S` is the total sugar uptake rate; and `\phi_N` is the nitrogen-growth phase proxy.

Volatilization is represented as an effective liquid-to-condenser transfer:

$$r_{loss,i} = \alpha_i K_i(T,E,S) q_{CO2} C_{L,i}.$$

The observation model uses

$$C_{wine,i}^{obs} = C_{total,i}^{obs} - C_{cond,i}^{obs}, \qquad C_{cond,i}^{obs} = C_{cond,i}.$$

Therefore the pilot condenser data make the `\alpha_i` loss parameters estimable as effective volatilization/capture coefficients.

In [1]:
from pathlib import Path
import pandas as pd
cwd = Path.cwd()
if (cwd / 'results' / 'calibration_estimability').exists():
    RESULTS = cwd / 'results' / 'calibration_estimability'
else:
    RESULTS = cwd / 'fermentation_model' / 'pilot_2025' / 'results' / 'calibration_estimability'
fit_summary = pd.read_csv(RESULTS / 'fit_summary.csv')
theta = pd.read_csv(RESULTS / 'theta_pilot_extended.csv', index_col=0)
estimability = pd.read_csv(RESULTS / 'parameter_estimability_current.csv')
weak = pd.read_csv(RESULTS / 'weak_directions_current.csv')
ranking = pd.read_csv(RESULTS / 'candidate_ranking_natural.csv')
selected = pd.read_csv(RESULTS / 'selected_campaign_hybrid.csv')
fit_summary

,fit,n_batches,mediums,parameters,n_parameters,success,status,message,nfev,initial_wsse,final_wsse,n_residuals,dof,wsse_per_residual,wsse_per_dof,l2_lambda,l2_parameters
0,pilot_core_l2_multistart_00,8.0,natural,"mu0,qN,betaG0,betaF0,qEG,qEF,iG,iE,Kd0,gammaG0...",11.0,True,3,`xtol` termination condition is satisfied.,6,2.791223e+04,27562.392291,828,817.0,33.287913,33.736098,0.35,"mu0,qN,betaG0,betaF0,qEG,qEF,iG,iE,Kd0,gammaG0..."
1,pilot_core_l2_multistart_01,8.0,natural,"mu0,qN,betaG0,betaF0,qEG,qEF,iG,iE,Kd0,gammaG0...",11.0,True,3,`xtol` termination condition is satisfied.,8,8.982938e+04,22471.946061,828,817.0,27.140031,27.505442,0.35,"mu0,qN,betaG0,betaF0,qEG,qEF,iG,iE,Kd0,gammaG0..."
2,pilot_secondary_v2_reduced_o2fixed,NaN,NaN,NaN,NaN,True,2,`ftol` termination condition is satisfied.,32,2.802082e+03,2430.302530,251,NaN,9.682480,NaN,NaN,NaN
3,pilot_aroma_multistart_00,NaN,NaN,NaN,NaN,True,2,`ftol` termination condition is satisfied.,12,1.563081e+06,4213.431374,297,NaN,14.186638,NaN,NaN,NaN
4,pilot_aroma_multistart_01,NaN,NaN,NaN,NaN,True,2,`ftol` termination condition is satisfied.,13,2.730195e+06,4212.776207,297,NaN,14.184432,NaN,NaN,NaN
5,pilot_aroma_multistart_02,NaN,NaN,NaN,NaN,True,2,`ftol` termination condition is satisfied.,21,9.425966e+05,4213.376763,297,NaN,14.186454,NaN,NaN,NaN


## Calibrated Parameter Vector

In [2]:
theta

,0
mu0,0.071110
sN,8.751446
qN,0.015763
qXG,0.081491
qXF,0.070933
betaG0,1.022129
sG,0.097455
betaF0,0.520241
sF,0.178295
qEG,1.312445


## Current-Data Estimability

The FIM is computed from log-parameter finite differences of the full residual vector. The approximate standard deviation is therefore in log-parameter space. Large `approx_95_multiplier` values indicate broad practical uncertainty. Parameters on active bounds are not considered reliable even when the local curvature appears high.

In [3]:
estimability.sort_values('std_log_approx', ascending=False)

,analysis,parameter,theta,std_log_approx,approx_95_multiplier,fim_diag,active_bound,classification
17,current_pilot,kAcStress,0.002616,358.693623,1.057654e+17,0.000000,False,weak_or_confounded
18,current_pilot,k_EA_growth,0.000639,12.927507,1.009523e+11,0.066205,False,weak_or_confounded
15,current_pilot,kAldRed,0.000010,5.906252,1.065376e+05,0.128085,False,weak_or_confounded
10,current_pilot,gammaF0,0.001959,2.058334,5.650529e+01,8.805469,False,weak_or_confounded
20,current_pilot,k_IAA_growth,0.009277,0.756979,4.409140e+00,11.375716,False,weak_but_actionable
19,current_pilot,k_EA_stationary,0.004711,0.607044,3.286441e+00,17.288910,False,moderate
6,current_pilot,iG,0.008233,0.186406,1.441025e+00,740.955468,False,well_estimated
23,current_pilot,k_EO_stationary,0.000160,0.183306,1.432297e+00,171.561476,False,well_estimated
22,current_pilot,k_EO_growth,0.000663,0.153957,1.352231e+00,258.547825,False,well_estimated
12,current_pilot,kPyrO2,0.282409,0.130897,1.292473e+00,1536.745887,False,well_estimated


## Weak Eigen-Directions

Weak directions identify combinations of parameters that the current pilot data do not separate well. These directions are more informative than single-parameter rankings when parameters are correlated.

In [4]:
weak

,analysis,weak_direction,eigenvalue,dominant_parameters,dominant_abs_loadings
0,current_pilot,1,7.178817e-15,"kAcStress, k_EA_growth, k_EA_stationary, kAldR...","1.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.00..."
1,current_pilot,2,5.966851e-03,"k_EA_growth, k_EA_stationary, alpha_EA_loss, k...","0.999, 0.039, 0.005, 0.001, 0.000, 0.000, 0.00..."
2,current_pilot,3,2.865587e-02,"kAldRed, kAcAld, iG, gammaF0, kAldS_N, betaF0,...","1.000, 0.009, 0.002, 0.002, 0.002, 0.001, 0.00..."
3,current_pilot,4,2.358005e-01,"gammaF0, gammaG0, k_IAA_growth, betaF0, iG, qE...","1.000, 0.027, 0.010, 0.007, 0.007, 0.003, 0.00..."
4,current_pilot,5,1.737094e+00,"k_IAA_growth, k_IAA_stationary, alpha_IAA_loss...","0.997, 0.065, 0.031, 0.011, 0.010, 0.004, 0.00..."
5,current_pilot,6,8.189951e+00,"k_EA_stationary, alpha_EA_loss, k_EA_growth, b...","0.987, 0.155, 0.037, 0.004, 0.002, 0.002, 0.00..."
6,current_pilot,7,1.797716e+01,"k_EO_stationary, k_EO_growth, alpha_EO_loss, b...","0.763, 0.631, 0.134, 0.026, 0.021, 0.015, 0.01..."
7,current_pilot,8,2.025731e+01,"iG, betaF0, qEF, iE, gammaG0, qEG, betaG0, k_E...","0.795, 0.355, 0.320, 0.279, 0.155, 0.143, 0.12..."


## Natural-Must Candidate Design Ranking

Candidate experiments are restricted to natural must. The manipulated variables are temperature setpoint profiles and nitrogen pulse timing/dose. No glucose, fructose, ethanol, or biomass injections are included in this pilot-scale design library.

In [5]:
cols = ['candidate', 'family', 'combined_logdet', 'combined_min_relative_eigenvalue', 'secondary_aroma_mean_var_reduction', 'secondary_aroma_worst_var_reduction', 'N_pulses_kg_m3', 'rationale']
ranking[cols].head(10)

,candidate,family,combined_logdet,combined_min_relative_eigenvalue,secondary_aroma_mean_var_reduction,secondary_aroma_worst_var_reduction,N_pulses_kg_m3,rationale
0,natural_pilot_midN_temperature_step,temperature_N,150.961925,1.103596e-07,0.369145,0.127762,54h:0.04,Temperature step with mid-growth nitrogen pert...
1,natural_pilot_high_rate_strip,co2_aroma,150.945219,1.101054e-07,0.369892,0.070231,30h:0.03,High-rate natural fermentation to excite CO2 s...
2,natural_pilot_cold_to_warm_earlyN,temperature_N,150.935091,1.148622e-07,0.354769,0.109550,30h:0.045,Cold start followed by warm transition and ear...
3,natural_pilot_two_step_N_ladder,N_timing,150.829205,1.072819e-07,0.349266,0.099016,30h:0.03; 78h:0.035,Two smaller nitrogen pulses to separate early ...
4,natural_pilot_reference_20C,reference,150.487297,1.148944e-07,0.391673,0.088477,NaN,Natural must warmer reference to increase rate...
5,natural_pilot_lateN_stationary_probe,N_timing,150.462320,1.071012e-07,0.357517,0.111879,80h:0.05,Late nitrogen addition to test stationary/grow...
6,natural_pilot_warm_to_cool_noN,temperature,150.390037,1.145248e-07,0.382908,0.048736,NaN,"Warm early phase to excite growth and CO2, the..."
7,natural_pilot_noN_dynamic_temperature,temperature,150.322301,1.143923e-07,0.397950,0.124206,NaN,Temperature-only perturbation for settings whe...
8,natural_pilot_reference_18C,reference,150.213210,1.149081e-07,0.399060,0.144845,NaN,Natural must reference at moderate temperature.
9,natural_pilot_low_temp_aroma_retention,aroma_retention,150.074912,1.053234e-07,0.359904,0.114962,54h:0.035,Cold profile to contrast aroma retention again...


## Selected Hybrid Campaign

The hybrid objective keeps D-optimality as the information-volume term and penalizes designs that leave very weak eigen-directions. This is the same practical logic used in the laboratory design fork.

In [6]:
selected[['campaign_order', 'candidate', 'family', 'campaign_logdet', 'campaign_min_relative_eigenvalue', 'secondary_aroma_mean_var_reduction', 'secondary_aroma_worst_var_reduction', 'N_pulses_kg_m3', 'rationale']]

,campaign_order,candidate,family,campaign_logdet,campaign_min_relative_eigenvalue,secondary_aroma_mean_var_reduction,secondary_aroma_worst_var_reduction,N_pulses_kg_m3,rationale
0,1,natural_pilot_cold_to_warm_earlyN,temperature_N,150.935091,1.148622e-07,0.354769,0.109550,30h:0.045,Cold start followed by warm transition and ear...
1,2,natural_pilot_high_rate_strip,co2_aroma,156.951545,1.447064e-07,0.479635,0.194498,30h:0.03,High-rate natural fermentation to excite CO2 s...
2,3,natural_pilot_low_temp_aroma_retention,aroma_retention,161.228952,1.568358e-07,0.546555,0.242726,54h:0.035,Cold profile to contrast aroma retention again...
3,4,natural_pilot_midN_temperature_step,temperature_N,164.689180,1.673093e-07,0.591963,0.273793,54h:0.04,Temperature step with mid-growth nitrogen pert...
4,5,natural_pilot_two_step_N_ladder,N_timing,167.551427,1.745888e-07,0.623624,0.294533,30h:0.03; 78h:0.035,Two smaller nitrogen pulses to separate early ...
5,6,natural_pilot_warm_to_cool_noN,temperature,170.032083,1.838583e-07,0.654827,0.323997,NaN,"Warm early phase to excite growth and CO2, the..."


## Generated Plots

Calibration fit plots are saved in `results/calibration_estimability/plots/fits`. Selected design input plots are saved in `results/calibration_estimability/plots/designs`.